# Experimento A/B en página de inicio

El objetivo de este proyecto es evaluar un **experimento A/B** realizado en una página de inicio (landing page) con versiónes **A y B** para apoyar una **decisión de negocio basada en datos**.

---

El archivo `landing_experiment.csv` contiene información de usuarios expuestos a dos versiones de la página de inicio (landing page) dentro del experimento A/B. Incluye región, dispositivo, fuente de tráfico, tipo de usuario, conversión y gasto.

El análisis sigue una lógica clara y progresiva:

1. 🔍 Explorar y validar los datos.

2. 💰 Comparar el **gasto promedio** por usuario entre la página A y B.

3. 🎯 Comparar la **tasa de conversión** entre la página A y B.

4. 🌐 Revisar **la relación entre la fuente de tráfico y la conversión**.

5. 👤 Revisar **la relación entre el tipo de usuario y la conversión**.

6. 📈 **Visualizar los resultados**: Respalda tus conclusiones mediante gráficos claros.

Se aplican **puebas estadísticas apropiados** para comparar las páginas y **recomendar qué versión es mejor**, justificando la decisión con datos.

## 🧩 Paso 1: Cargar y validar los datos

### 1.1 Carga de datos y vista rápida

In [1]:
#IMPORTS PRINCIPALES:
import pandas  as pd               # LECTURA, MANEJO Y TRANSFORMACIÓN DE DATAFRAMES.
import numpy   as np               # FUNCIONES MATEMÁTICAS RELEVANTES.
import matplotlib.pyplot as plt    # GRÁFICOS.
import seaborn as sns              # GRÁFICOS PRETTY.
from scipy.stats import ttest_ind  # Promedios (2 Grupos)
from scipy.stats import levene     # Varianzas
from statsmodels.stats.proportion import proportions_ztest  # Proporciones% Binarios
from scipy.stats import chi2_contingency #Chi cuadrado (categórica)

#CONFIGURACIONES GLOBALES:
import warnings                                                #MANEJO DE WARNINGS - ADVERTENCIAS
warnings.simplefilter(action='ignore', category=FutureWarning) #QUITAR WARNINGS MOLESTOS
pd.set_option('display.max_columns', None)                     #ELIMINA LIMITES DE PANDAS PARA MOSTRAR COLUMNAS
pd.set_option('display.max_rows', None)                        #ELIMINA LIMITES DE PANDAS PARA MOSTRAR FILAS
pd.set_option('display.max_colwidth', None)


In [2]:
# cargar archivo
# df = pd.read_csv('/datasets/landing_experiment.csv')

In [3]:
# Cargar el dataset y explorar datos
path = 'https://raw.githubusercontent.com/FithoGerardo/ab_experiment_landing_page/refs/heads/main/data/landing_experiment.csv'
df = pd.read_csv(path)

In [4]:
# Guardar copia
# df.to_csv('landing_experiment.csv', index=False, encoding='utf-8')

In [5]:
# copia df
dfc = df.copy() #dfc = df_copy

**Vista previa e información general del conjunto de datos**

In [6]:
display(dfc.head(5))

,user_id,date,landing,region,dispositivo,traffic_source,user_type,converted,gasto
0,26f3052e-8500-44ea-8fff-06de65258abb,2026-01-01,A,Norte,Mobile,Email,Recurrente,1,38.08
1,92378c09-4bbf-40c7-945e-82b84f392d22,2026-01-23,A,Occidente,Mobile,Organic,Nuevo,0,0.00
2,a4397360-40e5-45d6-a7ff-dcb4da2c9a1f,2026-01-01,B,Centro,Mobile,Organic,Nuevo,0,0.00
3,7ca3a26f-1e6c-44aa-9b09-b8cb01112956,2026-01-22,A,Centro,Mobile,Ads,Nuevo,0,0.00
4,8dc9593b-5b9c-479d-848b-a99493920419,2026-01-16,A,Sur,Mobile,Organic,Nuevo,0,0.00


In [7]:
display(dfc.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   user_id         40000 non-null  object 
 1   date            40000 non-null  object 
 2   landing         40000 non-null  object 
 3   region          40000 non-null  object 
 4   dispositivo     40000 non-null  object 
 5   traffic_source  40000 non-null  object 
 6   user_type       40000 non-null  object 
 7   converted       40000 non-null  int64  
 8   gasto           40000 non-null  float64
dtypes: float64(1), int64(1), object(7)
memory usage: 2.7+ MB


None

**Detección y corrección de inválidos y sentinels**

In [32]:
display(dfc.columns)
col_num = ['gasto']
col_cat = ['landing', 'region', 'dispositivo', 'traffic_source', 'user_type']
col_bina = ['converted']
col_date = ['date']

Index(['user_id', 'date', 'landing', 'region', 'dispositivo', 'traffic_source',
       'user_type', 'converted', 'gasto'],
      dtype='object')

In [50]:
# Verificar usuarios únicos
print("Columna user_id")
unique_users_num = dfc['user_id'].nunique()
print("Usuarios únicos: ", unique_users_num)
print()

# Resumen estadístico
print(f"Columnas numéricas: {col_num}")
for col in col_num:
  display(dfc[col].describe())
  print()

# Verificar categorías esperadas del experimento ( A y B).
print(f"Columnas categóricas: {col_cat}")
for col in col_cat:
  print(f"Columna: {col}")
  display(dfc[col].describe())
  print()
  display(dfc[col].value_counts().reset_index())
  print()

print(f"Columnas binarias: {col_bina}")
for col in col_bina:
  display(dfc[col].describe())
  print("\nResumen estadístico de usuarios que se convirtieron\n")
  display(dfc[col].value_counts().reset_index())
  print()

print(f"Columnas de fecha: {col_date}")
for col in col_date:
  display(dfc[col].describe())
  print()

Columna user_id
Usuarios únicos:  40000

Columnas numéricas: ['gasto']


,gasto
count,40000.000000
mean,9.325554
std,25.667986
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,303.680000



Columnas categóricas: ['landing', 'region', 'dispositivo', 'traffic_source', 'user_type']
Columna: landing


,landing
count,40000
unique,2
top,B
freq,20018


,landing,count
0,B,20018
1,A,19982


...

,converted
count,40000.00000
mean,0.14265
std,0.34972
min,0.00000
25%,0.00000
50%,0.00000
75%,0.00000
max,1.00000



Resumen estadístico de usuarios que se convirtieron



,converted,count
0,0,34294
1,1,5706



Columnas de fecha: ['date']


,date
count,40000
mean,2026-01-14 11:41:06
min,2026-01-01 00:00:00
25%,2026-01-07 00:00:00
50%,2026-01-14 00:00:00
75%,2026-01-21 00:00:00
max,2026-01-28 00:00:00


**Comentarios:**
- El dataset NO tiene valores ausentes.

- Convertir columna 'date' (object) a (date).
- 40,000 user_id diferentes
- landing = {B:	20018, A:	19982}
- Gasto Min  Max = 0, 303.68
- 5 region diferentes Top Norte 11166
- - **Norte, Centro, Sur, Occidente, Oriente**
- 2 dispositivo Top Mobile 24829
- - **Mobile, Desktop**
- 4 traffic_source Top Organic 17987
- - **Organic, Ads, Email, Referral**
- user_type = {Nuevo: 26033, Recurrente: 13967}
- converted = {0: 34294, 1: 5706}
- date unique 28 top 2026-01-24 freq 1512

**Convertir columna 'date' (object) a (date).**

In [47]:
# Convertir 'date' a fecha
dfc['date'] = pd.to_datetime(dfc['date'], errors='coerce')
# Confirmando cambios
print(f"Columnas de fecha: {col_date}")
for col in col_date:
  display(dfc[col].describe())
  print()
  display(dfc['date'].info())
  print()

# Identificar rango temporal del experimento
print("Fecha mínima:", dfc["date"].min())
print("Fecha máxima:", dfc["date"].max())

Columnas de fecha: ['date']


,date
count,40000
mean,2026-01-14 11:41:06
min,2026-01-01 00:00:00
25%,2026-01-07 00:00:00
50%,2026-01-14 00:00:00
75%,2026-01-21 00:00:00
max,2026-01-28 00:00:00



<class 'pandas.core.series.Series'>
RangeIndex: 40000 entries, 0 to 39999
Series name: date
Non-Null Count  Dtype         
--------------  -----         
40000 non-null  datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 312.6 KB


None


Fecha mínima: 2026-01-01 00:00:00
Fecha máxima: 2026-01-28 00:00:00


Columna 'date' abarca del día 1 - 28 de enero 2026

**Descripción del conjunto de datos**

El dataset contiene las siguientes columnas:

- `user_id` — Identificador único del usuario
- `date` — Fecha en la que el usuario fue expuesto a la página
- `landing` — Versión de la página mostrada al usuario
- `region` — Región geográfica del usuario
- `dispositivo` — Tipo de dispositivo utilizado por el usuario
- `traffic_source` — Canal por el que llegó el usuario
- `user_type` — Tipo de usuario según historial previo
- `converted` — Indica si el usuario realizó una conversión
- `gasto` — Monto gastado por el usuario (0 si no convirtió)

### 1.2 Análisis exploratorio y revisión de calidad de datos

Se identifican las variables clave del experimento A/B y se valida que estén bien definidas, completas y que sean consistentes.


In [51]:
dfc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   user_id         40000 non-null  object        
 1   date            40000 non-null  datetime64[ns]
 2   landing         40000 non-null  object        
 3   region          40000 non-null  object        
 4   dispositivo     40000 non-null  object        
 5   traffic_source  40000 non-null  object        
 6   user_type       40000 non-null  object        
 7   converted       40000 non-null  int64         
 8   gasto           40000 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(6)
memory usage: 2.7+ MB


**Comentarios:**
- Todas las columnas tienen valores esperados.
- Se hicieron correcciones en el apartado anterior.

## 💰 Paso 2: Comparar el gasto promedio por usuario (página A vs B)

Se evalua si existen diferencias estadísticamente significativas en el gasto promedio de los **usuarios que se convirtieron en clientes** entre la página A y la página B, para identificar qué versión genera **mayor valor económico** para el negocio.


In [52]:
# Gasto por versión (solo usuarios convertidos)
gasto_A = dfc[(dfc['landing'] == 'A') & (dfc['converted'] == 1)]['gasto']
gasto_B = dfc[(dfc['landing'] == 'B') & (dfc['converted'] == 1)]['gasto']

# Verificar cantidad de datos que tiene cada grupo
len(gasto_A), len(gasto_B)

(2512, 3194)

### Prueba t de Student t-test

**Hipótesis:**
- **Hipótesis nula (H₀):** El gasto promedio es igual en ambas páginas.
- **Hipótesis alternativa (H₁):** El gasto promedio es diferente entre las páginas.

In [ ]:
# Aplicar prueba



# Visualizar resultados
print(f"Estadístico : {...}")
print(f"Valor p: {...}")

### 📝 Conclusión e interpretación

**Decisión:**  
(¿Se rechaza o no la hipótesis nula?)

**Interpretación de negocio:**  
Explica con tus propias palabras qué indican estos resultados sobre el gasto promedio entre la página A y la página B.


---


## 📈 Paso 3: Comparar la tasa de conversión entre la página A y B

Se evalua si existen difere]ncias estadísticamente significativas en la **tasa de conversión** entre la página A y B, con el fin de identificar qué versión genera **mayor número de usuarios convertidos**.

### Prueba ...

**Hipótesis:**
- **Hipótesis nula (H₀):** ...
- **Hipótesis alternativa (H₁):** ...

In [ ]:
# Número de usuarios convertidos por página


# Total de usuarios por página


print("Usuarios convertidos por página:\n", ...)
print("\nTotal de usuarios por página:\n", ...)


In [ ]:
# Aplicar prueba



# Visualizar resultados
print(f"Estadístico : {...}")
print(f"Valor p: {...}")

### 📝 Conclusión e interpretación

**Decisión:**  
(¿Se rechaza o no la hipótesis nula?)

**Interpretación de negocio:**  
Explica qué indica el resultado sobre la tasa de conversión entre la página A y la página B.

## 🔗 Paso 4: Revisar la relación entre la fuente de tráfico y la conversión

Se analiza si existe una **asociación estadísticamente significativa** entre la **fuente de tráfico** (`traffic_source`) y la **conversión** (`converted`), para identificar qué canales generan más conversiones.

### Prueba ...

**Hipótesis:**
- **Hipótesis nula (H₀):** ...
- **Hipótesis alternativa (H₁):** ...

In [ ]:
# Aplicar prueba



### 📝 Conclusión e interpretación

**Decisión:**  
(¿Se rechaza o no la hipótesis nula?)

**Interpretación de negocio:**  
Explica qué indican los resultados considerando tanto las cantidades absolutas como las tasas de conversión.


## 👤 Paso 5: Revisar la relación entre el tipo de usuario y la conversión

Se analiza si existe una **asociación estadísticamente significativa** entre el **tipo de usuario** (`user_type`) y la **conversión** (`converted`), entendiendo que un usuario recurrente puede haber visitado antes sin necesariamente convertirse en cliente en esta ocasión.

El objetivo es identificar qué perfiles muestran mayor probabilidad de conversión dentro del contexto analizado.

### Prueba ...

**Hipótesis:**
- **Hipótesis nula (H₀):** ...
- **Hipótesis alternativa (H₁):** ...

In [ ]:
# Aplicar prueba



### 📝 Conclusión e interpretación

**Decisión:**  
(¿Se rechaza o no la hipótesis nula?)

**Interpretación de negocio:**  
Explica qué indica el resultado.

## 📊 Paso 6: Visualizar los resultados de variables categóricas

Se explora visualmente la relación entre variables categóricas (`traffic_source` y `user_type`) y la conversión, mostrando para cada categoría:
- la cantidad absoluta de usuarios que convirtieron y no convirtieron,
- la proporción de usuarios que convirtieron y no convirtieron.

Esto permite analizar tanto el impacto en volumen como la efectividad relativa de cada categoría y reforzar los resultados obtenidos en las pruebas estadísticas.

### Relación entre la fuente de tráfico y la conversión

✍️ **Comentario**: Haz doble clic en este bloque y complementa el gráfico con un breve texto que explique qué estamos viendo.

Comienza a escribir debajo de este texto, una vez escritas tus conclusiones, **elimina estas instrucciones** (de aqui hacia arriba de este bloque) para dejar solamente tus hallazgos.

✍️ **Comentario**: Haz doble clic en este bloque y complementa el gráfico con un breve texto que explique qué estamos viendo.

Comienza a escribir debajo de este texto, una vez escritas tus conclusiones, **elimina estas instrucciones** (de aqui hacia arriba de este bloque) para dejar solamente tus hallazgos.

### Relación entre el tipo de usuario y la conversión

✍️ **Comentario**: Haz doble clic en este bloque y complementa el gráfico con un breve texto que explique qué estamos viendo.

Comienza a escribir debajo de este texto, una vez escritas tus conclusiones, **elimina estas instrucciones** (de aqui hacia arriba de este bloque) para dejar solamente tus hallazgos.

✍️ **Comentario**: Haz doble clic en este bloque y complementa el gráfico con un breve texto que explique qué estamos viendo.

Comienza a escribir debajo de este texto, una vez escritas tus conclusiones, **elimina estas instrucciones** (de aqui hacia arriba de este bloque) para dejar solamente tus hallazgos.

## 🧩 Paso 7. Insight Ejecutivo para Stakeholders

Se traducen los hallazgos del análisis del experimento A/B en conclusiones accionables para el negocio, enfocadas en **versión de página, conversión, gasto promedio, canales de tráfico y tipo de usuario**.

**Preguntas a responder:**  
- ¿Qué página genera mayor conversión y gasto promedio?  
- ¿Qué canales de tráfico son más efectivos para generar conversiones?  
- ¿Existen diferencias significativas según el tipo de usuario?  
- ¿Qué recomendaciones se pueden tomar para optimizar la estrategia de marketing?


---

### 🌟 Insight Ejecutivo basado en el Experimento A/B

#### 🔍 **Comparación de página (A vs B)**  

**Gasto promedio por usuario que convirtió:**
- Observacion 1 aquí
- Observacion 2 aquí
- **Interpretación:**

<br>

**Tasa de conversión:**
- Observacion 1 aquí
- Observacion 2 aquí
- **Interpretación:**

---

#### 📊 **Segmentación por fuente de tráfico**
- Observacion aquí
- **Interpretación:**

 ---

#### 📊 **Segmentación por tipo de usuario**
- Observacion aquí
- **Interpretación:**

---

Las visualizaciones usadas respaldan los resultados estadísticos de pasos anteriores.

---

#### 💡 **Recomendaciones de negocio:**
- Recomendación aquí
-  Recomendación aquí